In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
%matplotlib inline
#Amitava Chakraborty - 11th May 2020
# Importing standard Qiskit libraries and configuring account
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister
from qiskit_backend_config import BasicAer, execute

from qiskit.compiler import transpile
from qiskit.quantum_info.operators.predicates import matrix_equal
from qiskit.circuit.library import HGate, RZGate, CXGate, CCXGate, XGate, SwapGate, DiagonalGate
from qiskit.visualization import plot_histogram, plot_bloch_multivector

# Diagonal Gate - Attach a diagonal gate to a circuit

In [ ]:
def _get_diag_gate_matrix(diag):
    return np.diagflat(diag)

phases = [[0, 0], [0, 0.8], [0, 0, 1, 1], [0, 1, 0.5, 1],
               (2 * np.pi * np.random.rand(2 ** 3)).tolist(),
               (2 * np.pi * np.random.rand(2 ** 4)).tolist(),
               (2 * np.pi * np.random.rand(2 ** 5)).tolist()]
print(phases)

In [ ]:
for phase in phases:
    diag = [np.exp(1j * ph) for ph in phase]
    print('diagonal : ',diag)
    num_qubits = int(np.log2(len(diag)))
    print('qubits : ',num_qubits)
    q = QuantumRegister(num_qubits)
    qc = QuantumCircuit(q)
    qc.append(DiagonalGate(diag), q[0:num_qubits])
    
    # Decompose the gate
    #qc = transpile(qc, basis_gates=['u1', 'u3', 'u2', 'cx', 'id'])
    
    # Simulate the decomposed gate
    simulator = BasicAer.get_backend('unitary_simulator')
    result = execute(qc, simulator).result()
    unitary = result.get_unitary(qc)
    unitary_desired = _get_diag_gate_matrix(diag)
    print(unitary)
    print('Desired Unitary : \n', unitary_desired)

In [ ]:
qc.draw('mpl')

In [ ]:
# diag (list) – list of the 2^k diagonal entries (for a diagonal gate on k qubits). Must contain at least two entries
# number of diagonal entries should be 2^k for k qubits
diag = [1, 1, -1, -1]  # entries must have |d| = 1
num_qubits = int(np.log2(len(diag)))
print('qubits : ',num_qubits)
q = QuantumRegister(num_qubits)
qc = QuantumCircuit(q)
h = HGate()
qc.append(h, qargs=[0])
qc.append(h, qargs=[1])
qc.append(DiagonalGate(diag), q[0:num_qubits])
qc.draw('mpl')

In [ ]:
simulator = BasicAer.get_backend('statevector_simulator')
result = execute(qc, simulator).result()
sv = result.get_statevector(qc)
print(sv)

In [ ]:
plot_bloch_multivector(sv)